# Разметка ЭКГ эксперимента 3

**Статус:** производитель кандидатов R-зубцов для записей РНЦХ. Детекция
перенесена из бывшего ноутбука 19, но её результат не считается проверенным без
ручного контроля.

ЭКГ задаёт временную привязку сердечных циклов и не локализует источник
биоимпедансного сигнала.


## Входы и выход

Ноутбук читает дыхательные sidecar-файлы 11.11 и исходные CSV, проверяет их
связь по SHA-256 и создаёт отдельные ЭКГ-sidecar-файлы. Модельные окна
электрической систолы не отождествляются с механическими или клапанными
событиями.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )
CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
RECORDINGS = {item["record_id"]: item for item in CONFIG["recordings"]}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_record(path):
    frame = pd.read_csv(path)
    frame.columns = [
        "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
        "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
    ]
    return frame

def sampling_frequency(frame):
    time = frame["time_s"].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("time_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction

from scipy.signal import butter, filtfilt, find_peaks

BREATHING_DIR = DERIVED_ROOT / "exp03" / "annotations" / "breathing"
OUT_DIR = DERIVED_ROOT / "exp03" / "annotations" / "ecg"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALGORITHM_VERSION = "exp03-ecg-rpeak-v1"


In [ ]:
def detect_rpeaks(ecg, fs_hz):
    high_hz = min(15.0, fs_hz / 2.0 - 1.0)
    if high_hz <= 5.0:
        raise ValueError("Частота дискретизации недостаточна для полосы 5–15 Гц")
    b, a = butter(
        2,
        [5.0 / (fs_hz / 2.0), high_hz / (fs_hz / 2.0)],
        btype="band",
    )
    filtered = filtfilt(b, a, np.asarray(ecg, dtype=float))
    filtered /= np.std(filtered) + 1e-9
    candidates, _ = find_peaks(
        filtered,
        distance=max(1, int(0.33 * fs_hz)),
        height=2.0,
        prominence=1.5,
    )
    raw = np.asarray(ecg, dtype=float)
    half_window = max(1, int(0.05 * fs_hz))
    refined = []
    for candidate in candidates:
        start = max(0, candidate - half_window)
        stop = min(len(raw), candidate + half_window)
        refined.append(start + int(np.argmax(raw[start:stop])))
    return np.asarray(sorted(set(refined)), dtype=int)


In [ ]:
annotations = []
for breathing_path in sorted(BREATHING_DIR.glob("*.json")):
    breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
    source_path = DATA_ROOT / breathing["input"]["relative_path"]
    input_sha256 = sha256_file(source_path)
    if input_sha256 != breathing["input"]["sha256"]:
        raise RuntimeError(
            f"Исходный файл изменился: {breathing['record_id']}"
        )
    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    peak_indices = detect_rpeaks(frame["ecg_v"].to_numpy(dtype=float), fs_hz)
    time = frame["time_s"].to_numpy(dtype=float)
    rpeaks_s = [float(time[index]) for index in peak_indices]
    output = {
        "schema_version": 1,
        "annotation_type": "ecg",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": breathing["record_id"],
        "input": {
            "relative_path": breathing["input"]["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_time_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "rpeaks_s": rpeaks_s,
        "upstream_breathing_qc": breathing["qc"]["status"],
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
        },
    }
    (OUT_DIR / f"{breathing['record_id']}.json").write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    annotations.append(output)
    print(output["record_id"], len(rpeaks_s), output["qc"]["status"])
print("Каталог:", OUT_DIR)


## Ручной контроль

Для каждой записи проверяются пропуски, ложные срабатывания, полярность и
участки с артефактами. Состояние дыхательной разметки фиксируется отдельно и
не принимает ЭКГ-разметку автоматически.


In [ ]:
CHECK_RECORD_ID = None
if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID.")
else:
    annotation = json.loads(
        (OUT_DIR / f"{CHECK_RECORD_ID}.json").read_text(encoding="utf-8")
    )
    source_path = DATA_ROOT / annotation["input"]["relative_path"]
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("Хеш исходного CSV изменился")
    frame = read_record(source_path)
    time = frame["time_s"].to_numpy(dtype=float)
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame["ecg_v"], color="0.35", linewidth=0.7)
    for r_time in annotation["rpeaks_s"]:
        axis.axvline(r_time, color="red", linewidth=0.6)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("ЭКГ, В")
    axis.set_title(
        f"{CHECK_RECORD_ID}: кандидаты R-зубцов; "
        f"QC={annotation['qc']['status']}"
    )
    plt.tight_layout()
    plt.show()
